In [1]:
#import library
import math
import pandas as pd

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
#dataset 4 dokumen
dokumen = [
    "Komputer digunakan untuk mengolah data.",
    "Jaringan komputer digunakan untuk berbagi data.",
    "Data komputer dapat disimpan dalam sistem.",
    "Sistem komputer membantu mengolah data."
]

for i, teks in enumerate(dokumen, 1):
    print(f"Dokumen {i}: {teks}")

Dokumen 1: Komputer digunakan untuk mengolah data.
Dokumen 2: Jaringan komputer digunakan untuk berbagi data.
Dokumen 3: Data komputer dapat disimpan dalam sistem.
Dokumen 4: Sistem komputer membantu mengolah data.


In [3]:
#membuat stopwprd remover dan  dan stemmer
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

In [4]:
#fungsi preprocessing
def preprocessing(text):
    #case folding
    text = text.lower()

    #menghilangkan angka, tanda baca, dan karakter khusus
    text = ''.join(
        karakter if karakter.isalpha() or karakter.isspace() else ' '
        for karakter in text
    )

    #tokenisasi
    tokens = text.split()

    #stopword removal
    text = ' '.join(tokens)
    text = stopword_remover.remove(text)

    tokens = text.split()

    return tokens

In [5]:
#tahap preprocessing
hasil_preprocessing = []

for teks in dokumen:
    hasil = preprocessing(teks)
    hasil_preprocessing.append(hasil)

for i, hasil in enumerate(hasil_preprocessing, 1):
    print(f"Dokumen {i}:")
    print(hasil)
    print()

Dokumen 1:
['komputer', 'digunakan', 'mengolah', 'data']

Dokumen 2:
['jaringan', 'komputer', 'digunakan', 'berbagi', 'data']

Dokumen 3:
['data', 'komputer', 'disimpan', 'sistem']

Dokumen 4:
['sistem', 'komputer', 'membantu', 'mengolah', 'data']



In [6]:
#tampilan sebelum dan sesudah preprocessing
for i in range(len(dokumen)):
    print("=" * 60)
    print(f"Dokumen {i + 1}")
    print("-" * 60)

    print("Sebelum preprocessing:")
    print(dokumen[i])

    print("\nSesudah preprocessing:")
    print(hasil_preprocessing[i])
    print()

Dokumen 1
------------------------------------------------------------
Sebelum preprocessing:
Komputer digunakan untuk mengolah data.

Sesudah preprocessing:
['komputer', 'digunakan', 'mengolah', 'data']

Dokumen 2
------------------------------------------------------------
Sebelum preprocessing:
Jaringan komputer digunakan untuk berbagi data.

Sesudah preprocessing:
['jaringan', 'komputer', 'digunakan', 'berbagi', 'data']

Dokumen 3
------------------------------------------------------------
Sebelum preprocessing:
Data komputer dapat disimpan dalam sistem.

Sesudah preprocessing:
['data', 'komputer', 'disimpan', 'sistem']

Dokumen 4
------------------------------------------------------------
Sebelum preprocessing:
Sistem komputer membantu mengolah data.

Sesudah preprocessing:
['sistem', 'komputer', 'membantu', 'mengolah', 'data']



In [7]:
#BAG-OF-WORDS
#Membuat vocabulary

vocabulary = sorted(set(
    token
    for dokumen_hasil in hasil_preprocessing
    for token in dokumen_hasil
))

print("Vocabulary:")
print(vocabulary)

Vocabulary:
['berbagi', 'data', 'digunakan', 'disimpan', 'jaringan', 'komputer', 'membantu', 'mengolah', 'sistem']


In [8]:
bow = []

for dokumen_hasil in hasil_preprocessing:
    baris = []

    for term in vocabulary:
        jumlah = dokumen_hasil.count(term)
        baris.append(jumlah)

    bow.append(baris)

In [9]:
bow_df = pd.DataFrame(
    bow,
    columns=vocabulary,
    index=[
        "Dokumen 1",
        "Dokumen 2",
        "Dokumen 3",
        "Dokumen 4"
    ]
)

print("Bag-of-Words:")
bow_df

Bag-of-Words:


,berbagi,data,digunakan,disimpan,jaringan,komputer,membantu,mengolah,sistem
Dokumen 1,0,1,1,0,0,1,0,1,0
Dokumen 2,1,1,1,0,1,1,0,0,0
Dokumen 3,0,1,0,1,0,1,0,0,1
Dokumen 4,0,1,0,0,0,1,1,1,1


In [10]:
#MENGHITUNG TF SECARA MANUAL
tf_manual = []

for dokumen_hasil in hasil_preprocessing:
    jumlah_kata = len(dokumen_hasil)

    baris = []

    for term in vocabulary:
        jumlah_term = dokumen_hasil.count(term)

        tf = jumlah_term / jumlah_kata

        baris.append(tf)

    tf_manual.append(baris)

tf_df = pd.DataFrame(
    tf_manual,
    columns=vocabulary,
    index=[
        "Dokumen 1",
        "Dokumen 2",
        "Dokumen 3",
        "Dokumen 4"
    ]
)

print("Matriks Term Frequency (TF):")
tf_df.round(4)

Matriks Term Frequency (TF):


,berbagi,data,digunakan,disimpan,jaringan,komputer,membantu,mengolah,sistem
Dokumen 1,0.0,0.25,0.25,0.00,0.0,0.25,0.0,0.25,0.00
Dokumen 2,0.2,0.20,0.20,0.00,0.2,0.20,0.0,0.00,0.00
Dokumen 3,0.0,0.25,0.00,0.25,0.0,0.25,0.0,0.00,0.25
Dokumen 4,0.0,0.20,0.00,0.00,0.0,0.20,0.2,0.20,0.20


In [11]:
#MENGHITUNG DF SECARA MANUAL
df_manual = {}

for term in vocabulary:
    jumlah_dokumen = 0

    for dokumen_hasil in hasil_preprocessing:
        if term in dokumen_hasil:
            jumlah_dokumen += 1

    df_manual[term] = jumlah_dokumen

df_df = pd.DataFrame(
    list(df_manual.items()),
    columns=["Term", "DF"]
)

print("Document Frequency (DF):")
df_df

Document Frequency (DF):


,Term,DF
0,berbagi,1
1,data,4
2,digunakan,2
3,disimpan,1
4,jaringan,1
5,komputer,4
6,membantu,1
7,mengolah,2
8,sistem,2


In [12]:
#MENGHITUNG IDF SECARA MANUAL
jumlah_dokumen = len(hasil_preprocessing)

idf_manual = {}

for term in vocabulary:
    idf = math.log(jumlah_dokumen / df_manual[term])

    idf_manual[term] = idf

idf_df = pd.DataFrame(
    list(idf_manual.items()),
    columns=["Term", "IDF"]
)

print("Inverse Document Frequency (IDF):")
idf_df.round(4)

Inverse Document Frequency (IDF):


,Term,IDF
0,berbagi,1.3863
1,data,0.0000
2,digunakan,0.6931
3,disimpan,1.3863
4,jaringan,1.3863
5,komputer,0.0000
6,membantu,1.3863
7,mengolah,0.6931
8,sistem,0.6931


In [13]:
# MENGHITUNG TF-IDF SECARA MANUAL
tfidf_manual = []

for i in range(len(hasil_preprocessing)):
    baris = []

    for j, term in enumerate(vocabulary):
        tf = tf_manual[i][j]
        idf = idf_manual[term]

        nilai_tfidf = tf * idf

        baris.append(nilai_tfidf)

    tfidf_manual.append(baris)

tfidf_manual_df = pd.DataFrame(
    tfidf_manual,
    columns=vocabulary,
    index=[
        "Dokumen 1",
        "Dokumen 2",
        "Dokumen 3",
        "Dokumen 4"
    ]
)

print("Matriks TF-IDF Manual:")
tfidf_manual_df.round(4)

Matriks TF-IDF Manual:


,berbagi,data,digunakan,disimpan,jaringan,komputer,membantu,mengolah,sistem
Dokumen 1,0.0000,0.0,0.1733,0.0000,0.0000,0.0,0.0000,0.1733,0.0000
Dokumen 2,0.2773,0.0,0.1386,0.0000,0.2773,0.0,0.0000,0.0000,0.0000
Dokumen 3,0.0000,0.0,0.0000,0.3466,0.0000,0.0,0.0000,0.0000,0.1733
Dokumen 4,0.0000,0.0,0.0000,0.0000,0.0000,0.0,0.2773,0.1386,0.1386


In [14]:
nomor_dokumen = 0

for j, term in enumerate(vocabulary):
    if tfidf_manual[nomor_dokumen][j] > 0:
        term_contoh = term
        break

jumlah_term = hasil_preprocessing[nomor_dokumen].count(term_contoh)
jumlah_kata = len(hasil_preprocessing[nomor_dokumen])

tf_contoh = jumlah_term / jumlah_kata
df_contoh = df_manual[term_contoh]
idf_contoh = idf_manual[term_contoh]
tfidf_contoh = tf_contoh * idf_contoh

print("Term                :", term_contoh)
print("Dokumen             : Dokumen 1")
print("Jumlah kemunculan   :", jumlah_term)
print("Jumlah seluruh kata :", jumlah_kata)
print("TF                  :", round(tf_contoh, 4))
print("DF                  :", df_contoh)
print("Jumlah dokumen (N)  :", jumlah_dokumen)
print("IDF                 :", round(idf_contoh, 4))
print("TF-IDF              :", round(tfidf_contoh, 4))

Term                : digunakan
Dokumen             : Dokumen 1
Jumlah kemunculan   : 1
Jumlah seluruh kata : 4
TF                  : 0.25
DF                  : 2
Jumlah dokumen (N)  : 4
IDF                 : 0.6931
TF-IDF              : 0.1733


In [15]:
#menggabungkan hasil preprocessing
dokumen_preprocessed = [
    ' '.join(hasil)
    for hasil in hasil_preprocessing
]

for i, teks in enumerate(dokumen_preprocessed, 1):
    print(f"Dokumen {i}: {teks}")

Dokumen 1: komputer digunakan mengolah data
Dokumen 2: jaringan komputer digunakan berbagi data
Dokumen 3: data komputer disimpan sistem
Dokumen 4: sistem komputer membantu mengolah data


In [16]:
#membuat TfidVectorizer

vectorizer = TfidfVectorizer(
    smooth_idf=False,
    norm=None
)

tfidf_sklearn = vectorizer.fit_transform(dokumen_preprocessed)

tfidf_sklearn_df = pd.DataFrame(
    tfidf_sklearn.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[
        "Dokumen 1",
        "Dokumen 2",
        "Dokumen 3",
        "Dokumen 4"
    ]
)

print("Matriks TF-IDF Scikit-Learn:")
tfidf_sklearn_df.round(4)

Matriks TF-IDF Scikit-Learn:


,berbagi,data,digunakan,disimpan,jaringan,komputer,membantu,mengolah,sistem
Dokumen 1,0.0000,1.0,1.6931,0.0000,0.0000,1.0,0.0000,1.6931,0.0000
Dokumen 2,2.3863,1.0,1.6931,0.0000,2.3863,1.0,0.0000,0.0000,0.0000
Dokumen 3,0.0000,1.0,0.0000,2.3863,0.0000,1.0,0.0000,0.0000,1.6931
Dokumen 4,0.0000,1.0,0.0000,0.0000,0.0000,1.0,2.3863,1.6931,1.6931


In [17]:
print("Perbandingan TF-IDF Manual dan Scikit-Learn")
print("=" * 50)

print("TF-IDF Manual:")
display(tfidf_manual_df.round(4))

print("\nTF-IDF Scikit-Learn:")
display(tfidf_sklearn_df.round(4))

Perbandingan TF-IDF Manual dan Scikit-Learn
TF-IDF Manual:


,berbagi,data,digunakan,disimpan,jaringan,komputer,membantu,mengolah,sistem
Dokumen 1,0.0000,0.0,0.1733,0.0000,0.0000,0.0,0.0000,0.1733,0.0000
Dokumen 2,0.2773,0.0,0.1386,0.0000,0.2773,0.0,0.0000,0.0000,0.0000
Dokumen 3,0.0000,0.0,0.0000,0.3466,0.0000,0.0,0.0000,0.0000,0.1733
Dokumen 4,0.0000,0.0,0.0000,0.0000,0.0000,0.0,0.2773,0.1386,0.1386



TF-IDF Scikit-Learn:


,berbagi,data,digunakan,disimpan,jaringan,komputer,membantu,mengolah,sistem
Dokumen 1,0.0000,1.0,1.6931,0.0000,0.0000,1.0,0.0000,1.6931,0.0000
Dokumen 2,2.3863,1.0,1.6931,0.0000,2.3863,1.0,0.0000,0.0000,0.0000
Dokumen 3,0.0000,1.0,0.0000,2.3863,0.0000,1.0,0.0000,0.0000,1.6931
Dokumen 4,0.0000,1.0,0.0000,0.0000,0.0000,1.0,2.3863,1.6931,1.6931


Pada perbandingan perhitungan TF-IDF secara manual maupun menggunakan scikit-learn(TfidfVectorizer) didapat hasil yang berbeda, hal  ini disebabkan karena karena cara menghitung TF manual berbeda dengan cara TfidfVectorizer menghitung TF.
Pada perhitungan manual digunakan 'tf = jumlah_term / jumlah_kata' yang artinya TF adalah frekuensi kata dibagi jumlah seluruh kata dalam dokumen.
Sedangkan, TfidfVectorizer secara default menggunakan jumlah kemunculan kata sebagai TF. 

In [18]:
#Mencari nilai TF-IDF tertinggi

for i in range(len(tfidf_manual_df)):
    baris = tfidf_manual_df.iloc[i]

    term_tertinggi = baris.idxmax()
    nilai_tertinggi = baris.max()

    print(f"Dokumen {i + 1}")
    print("Term dengan TF-IDF tertinggi :", term_tertinggi)
    print("Nilai TF-IDF                 :", round(nilai_tertinggi, 4))
    print()

Dokumen 1
Term dengan TF-IDF tertinggi : digunakan
Nilai TF-IDF                 : 0.1733

Dokumen 2
Term dengan TF-IDF tertinggi : berbagi
Nilai TF-IDF                 : 0.2773

Dokumen 3
Term dengan TF-IDF tertinggi : disimpan
Nilai TF-IDF                 : 0.3466

Dokumen 4
Term dengan TF-IDF tertinggi : membantu
Nilai TF-IDF                 : 0.2773



Analisis singkat:
Berdasarkan hasil perhitungan TF-IDF, setiap dokumen memiliki term dengan nilai TF-IDF tertinggi yang berbeda. Pada Dokumen 1, term “digunakan” memiliki nilai tertinggi sebesar 0,1733. Kemudian, Dokumen 2, term “berbagi” memiliki nilai tertinggi sebesar 0,2773. Selanjutnya, Dokumen 3, term “disimpan” memiliki nilai tertinggi sebesar 0,3466, sedangkan pada Dokumen 4 term “membantu” memiliki nilai tertinggi sebesar 0,2773. Nilai TF-IDF yang lebih tinggi menunjukkan bahwa term tersebut memiliki bobot yang lebih besar dalam dokumen dibandingkan term lainnya. Hal ini dapat terjadi karena term tersebut muncul dalam dokumen tetapi tidak terlalu banyak muncul pada dokumen lainnya.